In [1]:
# notebook: 03_cmvts_extension_branch1_branch2.ipynb
# ============================================================================
# CMVTS Extension — Branch 1 (real WoE-intensity sensitivity)
#                   Branch 2 (penetration-driven outcome, honest framing)
# ----------------------------------------------------------------------------
# Context from notebook 02:
#   - Predictor–outcome relationship is robust (Pearson -0.899 / Spearman -0.933)
#   - BUT the active-intensity profile came out uniform [.333,.333,.334] because
#     it was rank-tertiled, so realized JSD was effectively driven by the
#     active-penetration gap alone (source-vs-uniform JSD only 0.0234).
#
# Branch 1 asks: if we fill the active interior with the REAL Korean spend-
#   intensity shape (via monotonic WoE edges), does the correlation survive?
#   If yes -> the penetration simplification in Branch 2 loses no signal.
#
# Branch 2 then: formally treats realized JSD as an information-theoretic summary
#   of the active-penetration gap, and completes the predictor–outcome test on
#   that clean, observable basis (no Korea-interior assumption reused on targets).
# ============================================================================

import os
import numpy as np
import pandas as pd
from scipy import stats

# ----------------------------------------------------------------------------
# 0. CONFIG
# ----------------------------------------------------------------------------
CB_DIR   = "."
CB_FILE  = "202212_개인CB.csv"
FINDEX_CSV = "findex_microdata_2025_labelled_update112425.csv"

SOURCE_ECON  = "Korea, Rep."
TARGET_ECONS = ["Indonesia", "Thailand", "Viet Nam", "Philippines",
                "Bangladesh", "Cambodia", "Nepal", "Pakistan", "Lao PDR"]

YES = 1
CB_SENTINELS = [8888888.8, -9, -99999999]
CB_PRIMARY_SPEND = "C1M2B4W03"
CB_TARGET_LABEL  = "PERF1"
FINDEX_ACTIVITY  = "fin8"
N_ACTIVE_BINS    = 3

PREDICTOR_VARS = ["account_fin", "account_mob", "saved", "borrowed",
                  "receive_wages", "emp_in"]

# ----------------------------------------------------------------------------
# 1. HELPERS
# ----------------------------------------------------------------------------
def jsd(p, q, eps=1e-12):
    p = np.asarray(p, float) + eps; q = np.asarray(q, float) + eps
    p /= p.sum(); q /= q.sum(); m = 0.5 * (p + q)
    kl = lambda a, b: np.sum(a * np.log2(a / b))
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

def wshare(g, var, yes=YES):
    w = g["wgt"]; return w[g[var] == yes].sum() / w.sum()

def safe_cols(df, cols):
    present = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    if missing: print(f"  [warn] missing columns skipped: {missing}")
    return present

def macro_vector(g):
    cols = safe_cols(g, PREDICTOR_VARS)
    return np.array([wshare(g, v, YES) for v in cols], float)

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def report_corr(df, xcol, ycol, tag):
    r_p, p_p = stats.pearsonr(df[xcol], df[ycol])
    r_s, p_s = stats.spearmanr(df[xcol], df[ycol])
    print(f"[{tag}] Pearson {r_p:+.3f} (p={p_p:.4f}) | Spearman {r_s:+.3f} (p={p_s:.4f})")
    return r_p, r_s

# ----------------------------------------------------------------------------
# 2. Load data once
# ----------------------------------------------------------------------------
print("Loading CB source...")
cb = pd.read_csv(os.path.join(CB_DIR, CB_FILE), low_memory=False)
spend = cb[CB_PRIMARY_SPEND].replace(CB_SENTINELS, np.nan).fillna(0).astype(float)
has_label = CB_TARGET_LABEL in cb.columns
print("CB shape:", cb.shape, "| label present:", has_label)

print("Loading Findex...")
fx = pd.read_csv(FINDEX_CSV, low_memory=False)
kr_macro = macro_vector(fx[fx["economy"] == SOURCE_ECON])

# active mask on source
inactive_mask = spend <= 0
active_spend = spend[~inactive_mask]
p_active_src = float((~inactive_mask).mean())
print("Korea active share (non-zero spend): %.3f" % p_active_src)

# ============================================================================
# BRANCH 1 — Real Korean spend-intensity interior via monotonic WoE edges
# ============================================================================
print("\n" + "=" * 70)
print("BRANCH 1 — Real WoE-intensity interior (sensitivity check)")
print("=" * 70)

def monotonic_edges(x, y, max_bins=8, min_frac=0.05):
    """Monotonic coarse-classing edges on active spenders (bad-rate monotone)."""
    x = np.asarray(x, float); m = np.isfinite(x); x = x[m]
    y = np.asarray(y, float)[m] if y is not None else None
    edges = np.unique(np.quantile(x, np.linspace(0, 1, max_bins + 1)))
    if y is None or len(edges) < 3:
        return edges
    def rates(edges):
        idx = np.digitize(x, edges[1:-1], right=True)
        r = [y[idx == k].mean() if (idx == k).sum() else np.nan
             for k in range(len(edges) - 1)]
        c = [np.sum(idx == k) for k in range(len(edges) - 1)]
        return np.array(r), np.array(c)
    while len(edges) > 3:
        r, c = rates(edges)
        rr = r[~np.isnan(r)]
        mono = np.all(np.diff(rr) >= 0) or np.all(np.diff(rr) <= 0)
        small = np.where(c < min_frac * c.sum())[0]
        if mono and len(small) == 0:
            break
        drop = (small[0] + 1) if len(small) else (int(np.argmin(np.abs(np.diff(r)))) + 1)
        drop = max(1, min(drop, len(edges) - 2))
        edges = np.delete(edges, drop)
    return edges

# real interior profile from Korean active spenders
y_active = cb.loc[~inactive_mask, CB_TARGET_LABEL].astype(float).values if has_label else None
edges = monotonic_edges(active_spend.values, y_active, max_bins=N_ACTIVE_BINS + 4)
idx = np.digitize(active_spend.values, edges[1:-1], right=True)
n_real_bins = len(edges) - 1
real_profile = np.array([(idx == k).sum() for k in range(n_real_bins)], float)
real_profile /= real_profile.sum()
print("Monotonic edges:", np.round(edges, 1))
print("REAL Korean active-intensity profile:", np.round(real_profile, 4),
      f"({n_real_bins} bins)")

# source distribution with REAL interior
src_dist_b1 = np.concatenate([[1 - p_active_src], p_active_src * real_profile])
src_dist_b1 /= src_dist_b1.sum()
uni_b1 = np.ones_like(src_dist_b1) / len(src_dist_b1)
print("Source-vs-uniform JSD (Branch 1): %.4f  (was 0.0234 with uniform interior)"
      % jsd(src_dist_b1, uni_b1))

# targets: reuse REAL Korean interior for active mass (stated transfer assumption)
rows_b1 = []
for e in TARGET_ECONS:
    g = fx[fx["economy"] == e]
    if len(g) == 0: continue
    pa = wshare(g, FINDEX_ACTIVITY, YES)
    tdist = np.concatenate([[1 - pa], pa * real_profile]); tdist /= tdist.sum()
    rows_b1.append({"economy": e,
                    "X_macro_cos": round(cosine(kr_macro, macro_vector(g)), 4),
                    "Y_JSD_b1": round(jsd(src_dist_b1, tdist), 4)})
res_b1 = pd.DataFrame(rows_b1).sort_values("Y_JSD_b1").reset_index(drop=True)
print(res_b1.to_string(index=False))
report_corr(res_b1, "X_macro_cos", "Y_JSD_b1", "Branch 1")
print("Compare to notebook 02 (Pearson -0.899 / Spearman -0.933).")
print("If preserved -> the real interior adds no new structure beyond penetration,")
print("   which JUSTIFIES the Branch-2 penetration-only simplification.")

# ============================================================================
# BRANCH 2 — Penetration-driven outcome, stated honestly
# ============================================================================
print("\n" + "=" * 70)
print("BRANCH 2 — Penetration-gap outcome (clean, observable, no reused interior)")
print("=" * 70)
# Define the outcome directly and transparently as a 2-bin activity divergence:
#   target vs source over {inactive, active}. This is exactly what Findex can
#   observe for the target (binary activity), so NO Korean interior is imposed.
src_2bin = np.array([1 - p_active_src, p_active_src])
rows_b2 = []
for e in TARGET_ECONS:
    g = fx[fx["economy"] == e]
    if len(g) == 0: continue
    pa = wshare(g, FINDEX_ACTIVITY, YES)
    t_2bin = np.array([1 - pa, pa])
    rows_b2.append({"economy": e,
                    "target_active%": round(100 * pa, 1),
                    "X_macro_cos": round(cosine(kr_macro, macro_vector(g)), 4),
                    "Y_JSD_pen": round(jsd(src_2bin, t_2bin), 4)})
res_b2 = pd.DataFrame(rows_b2).sort_values("Y_JSD_pen").reset_index(drop=True)
print(res_b2.to_string(index=False))
report_corr(res_b2, "X_macro_cos", "Y_JSD_pen", "Branch 2")

# ----------------------------------------------------------------------------
# BRANCH 2b — cross-check: does Branch-1 and Branch-2 outcome rank-agree?
# ----------------------------------------------------------------------------
merged = res_b1.merge(res_b2, on=["economy", "X_macro_cos"])
r_cross, p_cross = stats.spearmanr(merged["Y_JSD_b1"], merged["Y_JSD_pen"])
print(f"\n[cross-check] Spearman(Y_b1, Y_pen) = {r_cross:+.3f} (p={p_cross:.4f})")
print("High agreement => the two outcome definitions are interchangeable in rank;")
print("Branch 2 keeps the same information with a simpler, more defensible outcome.")

# ----------------------------------------------------------------------------
# SUMMARY table for the paper (Branch 2 as primary, Branch 1 as robustness)
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PAPER-READY SUMMARY (Branch 2 primary)")
print("=" * 70)
summary = res_b2.merge(res_b1[["economy", "Y_JSD_b1"]], on="economy")
summary = summary.rename(columns={"Y_JSD_pen": "Y_primary(pen)",
                                  "Y_JSD_b1": "Y_robust(WoE)"})
print(summary.to_string(index=False))

Loading CB source...
CB shape: (3129036, 157) | label present: True
Loading Findex...
Korea active share (non-zero spend): 0.581

BRANCH 1 — Real WoE-intensity interior (sensitivity check)
Monotonic edges: [1.00000e+00 1.82100e+03 3.27300e+03 4.73000e+03 6.44800e+03 8.89400e+03
 1.35340e+04 1.66933e+05]
REAL Korean active-intensity profile: [0.1429 0.1429 0.1428 0.1428 0.1429 0.1428 0.1428] (7 bins)
Source-vs-uniform JSD (Branch 1): 0.0822  (was 0.0234 with uniform interior)
    economy  X_macro_cos  Y_JSD_b1
   Viet Nam       0.7575    0.0000
   Thailand       0.8178    0.0092
      Nepal       0.6526    0.0331
  Indonesia       0.7150    0.0706
    Lao PDR       0.6752    0.0751
   Cambodia       0.6265    0.0808
Philippines       0.5778    0.1806
 Bangladesh       0.5717    0.2107
   Pakistan       0.4891    0.2189
[Branch 1] Pearson -0.899 (p=0.0010) | Spearman -0.933 (p=0.0002)
Compare to notebook 02 (Pearson -0.899 / Spearman -0.933).
If preserved -> the real interior adds no new